Oracle AI Data Platform v1.0

Copyright © 2026, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# Delta UniForm metadata doctor

Checks every Delta **UniForm** table in one catalog and answers two questions about the Iceberg
metadata that UniForm generates (the files that ADW external tables and other Iceberg readers
depend on):

1. **Is it usable?** It must point at the table's *current* folder and reference files that
   exist. Iceberg metadata records **absolute paths** (per the Iceberg table specification),
   while Delta resolves its files relative to the table folder — so whenever a managed
   table's folder moves, the existing Iceberg metadata keeps referencing the previous
   location until it is regenerated. A **rename** is one such move: a managed table's folder
   is derived from its name (Hive-style semantics), so renaming the table moves the folder.
2. **Is it fresh?** By UniForm's design, conversion runs *after* each data commit,
   asynchronously, on the cluster that made the commit. When it cannot complete — for
   example, the cluster was stopped right after a write — the Iceberg view stays at the
   previous version until the next commit, which on a low-traffic table may be far away.

Neither situation is a platform quirk — both follow from the designs above. The doctor makes
the resulting states visible on a schedule, and repairable. Each table ends up in one state:

| State | Meaning | Repair (only with `MODE=repair`) |
|---|---|---|
| `HEALTHY` | usable and fresh | — |
| `LAGGING` | behind, for less than `MAX_LAG_MINUTES` | — (normal async behavior) |
| `STUCK` | behind, for longer than `MAX_LAG_MINUTES` | one small commit → triggers a new conversion |
| `BROKEN` | metadata missing, moved, or referencing missing files | move `metadata/` to a backup folder, then one small commit → full regeneration |
| `NOT_UNIFORM` | not a UniForm table | skipped |
| `UNREADABLE` | the check itself failed | reported, never touched |

**How to run it:** schedule this notebook as a workflow job and set the parameters on the task
(`CATALOG` is the only required one). The default `MODE=report` writes nothing anywhere and
makes the run **fail** when it finds a `BROKEN` table, so the workflow's failure alert is your
notification. See the README next to this notebook for details.

## 1 · Parameters

All inputs come from job parameters, so one notebook serves many catalogs and schedules.
Running interactively (outside a job), the defaults below apply.

In [ ]:
# oidlUtils is provided by the AIDP runtime - do not import it.
import re

def param(name, default):
    """Read one job parameter; fall back to the default when it is absent or empty.
    The lowercase spelling is tried too - job registrars often use it."""
    try:
        value = oidlUtils.parameters.getParameter(name, "")
        if value in ("", None):
            value = oidlUtils.parameters.getParameter(name.lower(), "")
    except NameError:          # not running as a job (interactive session without oidlUtils)
        value = ""
    return str(value) if value not in ("", None) else default

# The catalog to scan. Required - everything else has a sensible default.
CATALOG = param("CATALOG", "")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG):
    raise RuntimeError(f"Set the CATALOG job parameter to the catalog to scan. Got: {CATALOG!r}")

# report = read-only scan. repair = also fix STUCK and BROKEN tables.
MODE = param("MODE", "report").lower()

# How far behind the Iceberg conversion may be before we call the table STUCK.
# Conversion normally finishes seconds after a commit, so anything older than this
# means no conversion is coming on its own.
MAX_LAG_MINUTES = int(param("MAX_LAG_MINUTES", "30"))

# Comma-separated schema names to skip, e.g. staging areas.
EXCLUDE_SCHEMAS = {s.strip().lower() for s in param("EXCLUDE_SCHEMAS", "").split(",") if s.strip()}

# Safety cap: never repair more than this many tables in one run. If a whole catalog
# suddenly looks broken, something systemic happened - understand it before bulk-repairing.
MAX_REPAIRS = int(param("MAX_REPAIRS", "20"))

# After a repair commit, how long to wait for the conversion to catch up. Regeneration was
# measured at seconds up to 10k files and is independent of table bytes, so a repair that has
# not converged in 3 minutes has failed. A fat ceiling here multiplies: it is paid once per
# failing repair, up to MAX_REPAIRS times per run.
CONVERSION_WAIT_SECONDS = int(param("CONVERSION_WAIT_SECONDS", "180"))

# A STUCK table whose repair commit did not help has proven its conversions fail.
# Only with ESCALATE=true do we then rebuild its metadata like a BROKEN table.
ESCALATE = param("ESCALATE", "false").lower() == "true"

# When should the job run FAIL (so a scheduled workflow alerts)?
#   none   - never; the printed report is the only output
#   broken - on BROKEN or UNREADABLE tables, or a repair that did not converge  (default)
#   any    - also on STUCK tables
FAIL_ON = param("FAIL_ON", "broken").lower()

# Parallelism of the read-only scan.
SCAN_THREADS = int(param("SCAN_THREADS", "16"))

# Parallelism of the repair wave. Conversions serialize per cluster anyway (one at a time,
# measured ~5 s apiece back to back), so this only parallelizes the backups and the marker
# commits; the shared wait window below is what removes the per-table ceiling.
REPAIR_THREADS = int(param("REPAIR_THREADS", "8"))

if MODE not in ("report", "repair"):
    raise RuntimeError(f"MODE must be 'report' or 'repair', got {MODE!r}")
if FAIL_ON not in ("none", "broken", "any"):
    raise RuntimeError(f"FAIL_ON must be none, broken or any, got {FAIL_ON!r}")

print(f"doctor | catalog={CATALOG} mode={MODE} max_lag={MAX_LAG_MINUTES}min "
      f"fail_on={FAIL_ON} max_repairs={MAX_REPAIRS} escalate={ESCALATE}")

## 2 · Small helpers

Reading one table costs two catalog calls (`SHOW TBLPROPERTIES`, `DESCRIBE EXTENDED`) plus two
flat folder listings and one small-file read, all through the Hadoop filesystem — **no Spark
job per table**. A `DESCRIBE HISTORY` or `spark.read.text` would each pay a Spark-scheduler
round trip per table, which at thousands of tables dominates the scan. `DESCRIBE DETAIL` is
deliberately **not** used — it fails on UniForm tables on AIDP.

In [ ]:
import json
import time
import datetime
import decimal
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql import types as T

# Hadoop filesystem handles - this is how the notebook lists and reads the table's files in
# Object Storage. It uses the cluster's own storage identity; no credential is configured here.
jvm = spark._jvm
HadoopPath = jvm.org.apache.hadoop.fs.Path
hadoop_conf = spark._jsc.hadoopConfiguration()

def filesystem(path):
    return HadoopPath(path).getFileSystem(hadoop_conf)

def path_exists(path):
    return bool(filesystem(path).exists(HadoopPath(path)))

def read_small_file(path):
    """One small text file, read straight through the Hadoop filesystem. spark.read.text
    would schedule a Spark job for a 10 KB file - at thousands of tables that is the scan."""
    stream = filesystem(path).open(HadoopPath(path))
    try:
        return jvm.org.apache.commons.io.IOUtils.toString(stream, "UTF-8")
    finally:
        stream.close()

def qualified(*name_parts):
    """catalog.schema.table, refusing anything that is not a plain identifier.
    The names are joined WITHOUT backticks: identifiers matching the pattern need none, and
    some commands (SHOW TBLPROPERTIES among them) resolve a quoted catalog name differently."""
    for part in name_parts:
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", part):
            raise ValueError(f"not a plain identifier: {part!r}")
    return ".".join(name_parts)

def table_location(full_name):
    """The table's folder in Object Storage, from DESCRIBE EXTENDED."""
    for row in spark.sql(f"DESCRIBE EXTENDED {full_name}").collect():
        if (row[0] or "").strip().lower() == "location":
            return (row[1] or "").rstrip("/")
    raise ValueError("Location not found in DESCRIBE EXTENDED")

def table_properties(full_name):
    rows = spark.sql(f"SHOW TBLPROPERTIES {full_name}").collect()
    return {row[0]: row[1] for row in rows}

def is_uniform(properties):
    return "iceberg" in (properties.get("delta.universalFormat.enabledFormats") or "")

def delta_log_versions(location):
    """{commit version: commit time} from ONE flat listing of _delta_log. A commit file's
    modification time is exactly what DESCRIBE HISTORY reports as the commit timestamp, at a
    fraction of the cost: no Spark job, no log replay. Commit files are the 20-digit .json
    names; checkpoints, compacted logs and CRCs fall out of the name test."""
    entries = {}
    for status in filesystem(location).listStatus(HadoopPath(location + "/_delta_log")):
        name = status.getPath().getName()
        if len(name) == 25 and name.endswith(".json") and name[:20].isdigit():
            entries[int(name[:20])] = datetime.datetime.fromtimestamp(
                status.getModificationTime() / 1000.0, datetime.timezone.utc)
    if not entries:
        raise ValueError("no commit files under _delta_log - not a Delta table?")
    return entries

def latest_delta_commit(location):
    """(version, timestamp) of the table's newest Delta commit, from the log listing."""
    versions = delta_log_versions(location)
    newest = max(versions)
    return newest, versions[newest]

def minutes_since(timestamp):
    if timestamp is None:
        return None
    now = datetime.datetime.now(timestamp.tzinfo)
    return (now - timestamp).total_seconds() / 60.0

print("helpers ready")

## 3 · Reading the Iceberg side

UniForm writes `metadata/vN.metadata.json` files inside the table folder. Each one records:

* `location` — the table folder the metadata was generated for (an **absolute** path, per the
  Iceberg table specification — this is why a rename calls for regeneration);
* the current snapshot, whose `manifest-list` file must exist;
* the property `delta-version` — **which Delta commit this metadata was converted from.**
  This is the completion signal: conversion is done when it has caught up with Delta.

In [ ]:
def read_iceberg_metadata(location):
    """Parse the newest vN.metadata.json of a table, or return None if there is none."""
    fs = filesystem(location)
    metadata_dir = HadoopPath(location + "/metadata")
    if not fs.exists(metadata_dir):
        return None
    # Find the vN.metadata.json with the highest N.
    newest_name, newest_n = None, -1
    for status in fs.listStatus(metadata_dir):
        name = status.getPath().getName()
        if name.startswith("v") and name.endswith(".metadata.json"):
            try:
                n = int(name[1:-len(".metadata.json")])
            except ValueError:
                continue                       # a stray file, not the table's metadata
            if n > newest_n:
                newest_name, newest_n = name, n
    if newest_name is None:
        return None
    content = read_small_file(f"{location}/metadata/{newest_name}")
    metadata = json.loads(content)

    # Pull out the three things the health check needs.
    snapshots = metadata.get("snapshots", [])
    current = next((s for s in snapshots
                    if s["snapshot-id"] == metadata.get("current-snapshot-id")), None)
    converted = (metadata.get("properties") or {}).get("delta-version")
    return {
        "file": newest_name,
        "location": (metadata.get("location") or "").rstrip("/"),
        "has_current_snapshot": current is not None,
        "manifest_list": current.get("manifest-list") if current else None,
        "converted_delta_version": int(converted) if converted is not None else None,
    }

## 4 · Classifying one table

Structure first, freshness second. One ordering rule keeps the check safe against concurrent
writers: read the **Delta version before the Iceberg metadata**. A write that lands in between
can then only make Iceberg look *ahead* of what we saw — never falsely behind — so every
comparison below uses `>=`.

In [ ]:
def check_table(schema, table):
    """Examine one table and return a dict with at least: schema, table, full, state."""
    full = qualified(CATALOG, schema, table)
    result = {"schema": schema, "table": table, "full": full}
    try:
        if not is_uniform(table_properties(full)):
            result["state"] = "NOT_UNIFORM"
            return result

        location = table_location(full)
        delta_log = delta_log_versions(location)                    # Delta first (see above)
        delta_version = max(delta_log)
        last_commit_at = delta_log[delta_version]
        iceberg = read_iceberg_metadata(location)
        result["location"] = location
        result["delta_version"] = delta_version

        # --- no Iceberg metadata at all -------------------------------------------------------
        if iceberg is None:
            age = minutes_since(last_commit_at)
            if age is not None and age < MAX_LAG_MINUTES:
                # The table is young or was just written; the first conversion may still come.
                result["state"] = "LAGGING"
                result["why"] = f"never converted yet, last commit {age:.0f} min ago"
            else:
                result["state"], result["kind"] = "BROKEN", "MISSING"
                result["why"] = "UniForm is enabled but no Iceberg metadata exists"
            return result

        result["converted_delta_version"] = iceberg["converted_delta_version"]

        # --- structure: is the metadata usable at all? ----------------------------------------
        if iceberg["location"] != location:
            # The rename case: the folder moved and the existing metadata still references the
            # previous path. Metadata generation is incremental, so this persists until the
            # metadata is regenerated at the current location - which is what the repair does.
            result["state"], result["kind"] = "BROKEN", "MOVED"
            result["why"] = f"metadata points at {iceberg['location']}"
        elif not iceberg["has_current_snapshot"]:
            # Metadata without a current snapshot cannot serve a read; same repair as DAMAGED.
            result["state"], result["kind"] = "BROKEN", "DAMAGED"
            result["why"] = "metadata records no current snapshot"
        elif iceberg["manifest_list"] and not path_exists(iceberg["manifest_list"]):
            result["state"], result["kind"] = "BROKEN", "DAMAGED"
            result["why"] = "current manifest list file is missing"
        elif iceberg["converted_delta_version"] is None:
            result["state"], result["kind"] = "BROKEN", "DAMAGED"
            result["why"] = "metadata does not record a delta-version"

        # --- freshness: has conversion caught up with Delta? ----------------------------------
        elif iceberg["converted_delta_version"] >= delta_version:
            result["state"] = "HEALTHY"
        else:
            # Behind. How long has the OLDEST unconverted commit been waiting?
            # the listing already read is the history; a cleaned-up commit falls back below
            first_unconverted = delta_log.get(iceberg["converted_delta_version"] + 1)
            age = minutes_since(first_unconverted or last_commit_at)
            result["lag_minutes"] = round(age, 1) if age is not None else None
            result["why"] = (f"converted up to v{iceberg['converted_delta_version']}, "
                             f"Delta is at v{delta_version}")
            if age is not None and age < MAX_LAG_MINUTES:
                result["state"] = "LAGGING"     # normal async behavior, no action
            else:
                result["state"] = "STUCK"       # no conversion is coming on its own
    except Exception as error:
        # One unreadable table must not blind the doctor to the rest of the catalog.
        # Spark wraps Java errors in a generic first line; keep the informative tail too.
        result["state"] = "UNREADABLE"
        result["why"] = " | ".join(l.strip() for l in str(error).splitlines()[:4] if l.strip())[:300]
    return result

## 5 · Scan the catalog and report

In [ ]:
def list_schemas():
    rows = spark.sql(f"SHOW NAMESPACES IN {qualified(CATALOG)}").collect()
    return [row[0] for row in rows if row[0].lower() not in EXCLUDE_SCHEMAS]

def list_tables(schema):
    """Table names in one schema. Spark's SHOW TABLES also returns views, so subtract them."""
    rows = spark.sql(f"SHOW TABLES IN {qualified(CATALOG, schema)}").collect()
    tables = [row["tableName"] for row in rows]
    try:
        view_rows = spark.sql(f"SHOW VIEWS IN {qualified(CATALOG, schema)}").collect()
        views = {str(row["viewName"]).lower() for row in view_rows}
        tables = [t for t in tables if t.lower() not in views]
    except Exception:
        pass                                   # no view support in this catalog - nothing to subtract
    return tables

started = time.time()
pairs, schema_failures = [], []
for schema in list_schemas():
    try:
        for table in list_tables(schema):
            pairs.append((schema, table))
    except Exception as error:
        # One schema that cannot be listed must not blind the doctor to the rest.
        schema_failures.append({"schema": schema, "table": "*", "full": f"{CATALOG}.{schema}.*",
                                "state": "UNREADABLE",
                                "why": str(error).splitlines()[0][:160]})
print(f"scanning {len(pairs)} table(s) in {CATALOG} with {SCAN_THREADS} thread(s)...")

with ThreadPoolExecutor(max_workers=SCAN_THREADS) as pool:
    RESULTS = list(pool.map(lambda pair: check_table(pair[0], pair[1]), pairs))
RESULTS.extend(schema_failures)

def tables_in(state):
    return [r for r in RESULTS if r["state"] == state]

COUNTS = {state: len(tables_in(state))
          for state in ("HEALTHY", "LAGGING", "STUCK", "BROKEN", "NOT_UNIFORM", "UNREADABLE")}
print(f"scan done in {time.time() - started:.1f}s | {COUNTS}")

# Print one line per table that deserves attention.
for r in RESULTS:
    if r["state"] in ("LAGGING", "STUCK", "BROKEN", "UNREADABLE"):
        kind = f"/{r['kind']}" if r.get("kind") else ""
        print(f"   {r['state'] + kind:<16} {r['schema']}.{r['table']} :: {r.get('why', '')}")

## 6 · Repair (does nothing in `report` mode)

Every repair is the same simple recipe:

1. If the metadata is unusable (`MOVED` / `DAMAGED`), move the whole `metadata/` folder to a
   backup next to the table — **never delete it**. With no previous Iceberg state, the next
   conversion starts from scratch and writes correct paths for the table's current folder.
2. Make a small **data** commit so the converter runs: a marker row is inserted and deleted
   again, leaving the data exactly as it was. (A table-properties-only commit does not trigger
   a conversion — measured on this platform — so a data commit it is.)
3. Wait until the table is healthy: metadata at the right location, files present, and
   `delta-version` caught up with the commit we just made.

Repairs run as a **wave**: step 1 and 2 for every table first (threaded), then one shared
wait window for all of them. A successful conversion lands in seconds; the full
`CONVERSION_WAIT_SECONDS` is only ever paid when a repair truly failed — and with the shared
window it is paid **once per wave**, not once per table. Conversions serialize per cluster
(one at a time), so the tables converge back to back inside the window; `ESCALATE` adds at
most one more wave.

In [ ]:
MARKER_TEXT = "__uniform_doctor__"
MARKER_NUMBER = -999000111

def is_healthy_now(location, min_delta_version):
    """The post-repair success test - same rules as the scan."""
    try:
        iceberg = read_iceberg_metadata(location)
        if iceberg is None or iceberg["location"] != location:
            return False
        if not iceberg["has_current_snapshot"] or not iceberg["manifest_list"]:
            return False
        if not path_exists(iceberg["manifest_list"]):
            return False
        converted = iceberg["converted_delta_version"]
        return converted is not None and converted >= min_delta_version
    except Exception:
        return False

def move_metadata_to_backup(location):
    """Move <table>/metadata to <schema folder>/_uniform_doctor_bkp/<table>__metadata_<ts>.
    The backup sits OUTSIDE the table folder, so VACUUM never sees it, and restoring it
    is a single rename. Returns the backup path."""
    fs = filesystem(location)
    source = HadoopPath(location + "/metadata")
    if not fs.exists(source):
        return None
    schema_folder, table_folder = location.rsplit("/", 1)
    # Human-readable UTC stamp: repairing the same table more than once never collides.
    stamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.gmtime()) + "_utc"
    backup = HadoopPath(f"{schema_folder}/_uniform_doctor_bkp/{table_folder}__metadata_{stamp}")
    fs.mkdirs(backup.getParent())
    if not fs.rename(source, backup):
        raise RuntimeError("could not move metadata/ aside")
    return backup.toString()

def restore_metadata_backup(location, backup):
    """Undo move_metadata_to_backup with one rename, so a failed repair leaves the table
    exactly as the run found it."""
    fs = filesystem(location)
    source = HadoopPath(location + "/metadata")
    if backup and fs.exists(HadoopPath(backup)) and not fs.exists(source):
        return bool(fs.rename(HadoopPath(backup), source))
    return False

def marker_commit(full_name):
    """Insert one marker row and delete it again - the data ends exactly as it was.
    The marker column is chosen so the value cannot overflow: a string column first, then
    int/bigint, then double, then a decimal wide enough for the 9-digit marker.
    smallint/tinyint are never used - the marker does not fit them."""
    schema = spark.table(full_name).schema
    candidates = []
    for wanted_types, value, literal in [
            ((T.StringType,), MARKER_TEXT, f"'{MARKER_TEXT}'"),
            ((T.LongType, T.IntegerType), MARKER_NUMBER, str(MARKER_NUMBER)),
            ((T.DoubleType,), float(MARKER_NUMBER), str(float(MARKER_NUMBER)))]:
        for field in schema.fields:
            if isinstance(field.dataType, wanted_types):
                candidates.append((field.name, value, literal))
    for field in schema.fields:
        # A decimal column qualifies when its integer digits fit the 9-digit marker.
        if isinstance(field.dataType, T.DecimalType) and \
                field.dataType.precision - field.dataType.scale >= 10:
            candidates.append((field.name, decimal.Decimal(MARKER_NUMBER), str(MARKER_NUMBER)))
    marker_column, marker_value, marker_literal = None, None, None
    for name, value, literal in candidates:
        # The DELETE below removes EVERY row equal to the marker, so a column whose real data
        # already contains the marker value must not be used - try the next candidate. An
        # existence probe, not a COUNT: it stops at the first match instead of scanning it all.
        hit = spark.sql(f"SELECT 1 FROM {full_name} WHERE `{name}` = {literal} LIMIT 1").first()
        if hit is None:
            marker_column, marker_value, marker_literal = name, value, literal
            break
    if marker_column is None:
        raise RuntimeError("no safe marker column (string/int/bigint/double/wide decimal) "
                           "is free of the marker value")

    def filler(field):
        """NULL where allowed; a neutral value for NOT NULL columns."""
        if field.nullable:
            return None
        if isinstance(field.dataType, T.StringType):   return ""
        if isinstance(field.dataType, T.BooleanType):  return False
        if isinstance(field.dataType, T.DecimalType):  return decimal.Decimal(0)
        if isinstance(field.dataType, T.DateType):     return datetime.date(1970, 1, 1)
        if isinstance(field.dataType, T.TimestampType): return datetime.datetime(1970, 1, 1)
        if isinstance(field.dataType, (T.DoubleType, T.FloatType)): return 0.0
        return 0

    row = tuple(marker_value if f.name == marker_column else filler(f) for f in schema.fields)
    spark.createDataFrame([row], schema=schema).writeTo(full_name).append()
    predicate = f"`{marker_column}` = {marker_literal}"
    for attempt in (1, 2, 3):                     # a concurrent writer can make one DELETE lose
        try:
            spark.sql(f"DELETE FROM {full_name} WHERE {predicate}")
            return
        except Exception:
            if attempt == 3:
                raise RuntimeError(f"marker DELETE failed 3x - remove the row manually: "
                                   f"DELETE FROM {full_name} WHERE {predicate}")
            time.sleep(5)

def prepare_repair(result):
    """Steps 1 and 2 of the recipe for one table: unusable metadata aside, marker commit.
    Returns the report carrying the Delta version the conversion must reach."""
    full, location = result["full"], result["location"]
    report = {"table": f"{result['schema']}.{result['table']}", "full": full,
              "location": location, "state": result["state"], "kind": result.get("kind"),
              "fixed": False}
    try:
        if result.get("kind") in ("MOVED", "DAMAGED"):
            report["backup"] = move_metadata_to_backup(location)
        # A data commit is what triggers a conversion (a table-properties-only commit does
        # not - measured on this platform).
        marker_commit(full)
        report["target_version"] = latest_delta_commit(location)[0]
    except Exception as error:
        report["why"] = str(error).splitlines()[0][:200]
        # The commit did not happen, so no regeneration is coming: put the metadata back and
        # leave the table exactly as the run found it.
        if report.get("backup") and restore_metadata_backup(location, report["backup"]):
            report["restored_backup"] = report.pop("backup")
    return report

def await_repairs(reports, seconds, via):
    """Step 3, shared: ONE wait window for every pending repair. A table is marked fixed the
    moment its metadata catches up, so a successful wave finishes in seconds; the full window
    is only consumed by repairs that truly failed - and once per wave, not once per table."""
    deadline = time.time() + seconds
    pending = [r for r in reports if not r["fixed"] and "target_version" in r]
    while pending and time.time() < deadline:
        time.sleep(10)
        still_pending = []
        for report in pending:
            if is_healthy_now(report["location"], report["target_version"]):
                report["fixed"], report["via"] = True, via
            else:
                still_pending.append(report)
        pending = still_pending
    for report in pending:
        report["why"] = "conversion did not catch up in time"

REPAIR_REPORTS = []
to_repair = tables_in("BROKEN") + tables_in("STUCK")      # broken first
if MODE != "repair":
    print(f"report mode - {len(to_repair)} table(s) would be repaired. Set MODE=repair to act.")
elif not to_repair:
    print("repair mode - nothing to repair.")
else:
    if len(to_repair) > MAX_REPAIRS:
        print(f"!! {len(to_repair)} repairable tables exceed MAX_REPAIRS={MAX_REPAIRS} - "
              f"repairing the first {MAX_REPAIRS} only. If a whole catalog broke at once, "
              f"find out why before raising the cap.")
    batch = to_repair[:MAX_REPAIRS]
    with ThreadPoolExecutor(max_workers=min(REPAIR_THREADS, len(batch))) as pool:
        REPAIR_REPORTS = list(pool.map(prepare_repair, batch))
    await_repairs(REPAIR_REPORTS, CONVERSION_WAIT_SECONDS, "marker commit")

    # A STUCK table whose marker commit did not help has proven its conversions fail.
    # Rebuild its metadata like a BROKEN table - but only when explicitly allowed, and only
    # for tables whose marker commit itself succeeded. One more wave, one more window.
    escalatable = [r for r in REPAIR_REPORTS
                   if not r["fixed"] and r["state"] == "STUCK" and ESCALATE
                   and "target_version" in r and "backup" not in r]
    if escalatable:
        def escalate(report):
            try:
                report.pop("why", None)
                report["backup"] = move_metadata_to_backup(report["location"])
                marker_commit(report["full"])
                report["target_version"] = latest_delta_commit(report["location"])[0]
            except Exception as error:
                report["why"] = str(error).splitlines()[0][:200]
                # A STUCK table was still READABLE: escalation must never leave it without
                # metadata. Put the backup back, and leave the wait window too - no commit
                # landed, so nothing can catch up and waiting would only mask this error.
                if report.get("backup") and restore_metadata_backup(report["location"], report["backup"]):
                    report["restored_backup"] = report.pop("backup")
                report.pop("target_version", None)
        with ThreadPoolExecutor(max_workers=min(REPAIR_THREADS, len(escalatable))) as pool:
            list(pool.map(escalate, escalatable))
        await_repairs(escalatable, CONVERSION_WAIT_SECONDS, "rebuild after escalation")

    for report in REPAIR_REPORTS:
        outcome = f"FIXED ({report['via']})" if report["fixed"] else f"NOT FIXED - {report.get('why', '')}"
        print(f"   {report['table']:<40} {outcome}")
        if report.get("backup"):
            print(f"      backup: {report['backup']}")

## 7 · Verdict

A monitoring job must fail loudly when it finds what it watches for — `FAIL_ON` tunes that.

In [ ]:
fixed_tables = {report["table"] for report in REPAIR_REPORTS if report["fixed"]}

final_counts = dict.fromkeys(COUNTS, 0)
for r in RESULTS:
    if f"{r['schema']}.{r['table']}" in fixed_tables:
        final_counts["HEALTHY"] += 1
    else:
        final_counts[r["state"]] += 1

failed_repairs = [report["table"] for report in REPAIR_REPORTS if not report["fixed"]]

print(f"\n===== VERDICT ===== catalog={CATALOG} mode={MODE}")
print("   " + json.dumps(final_counts))
if failed_repairs:
    print("   repair failures: " + ", ".join(failed_repairs))

# One table = one problem, even when it appears both as BROKEN and as a failed repair.
problem_tables = {f"{r['schema']}.{r['table']}" for r in RESULTS
                  if r["state"] in ("BROKEN", "UNREADABLE")
                  and f"{r['schema']}.{r['table']}" not in fixed_tables}
problem_tables |= set(failed_repairs)
if FAIL_ON == "any":
    problem_tables |= {f"{r['schema']}.{r['table']}" for r in RESULTS
                       if r["state"] == "STUCK" and f"{r['schema']}.{r['table']}" not in fixed_tables}
problems = len(problem_tables)
if FAIL_ON != "none" and problems > 0:
    raise RuntimeError(f"{problems} table(s) need attention in {CATALOG}: "
                       f"broken={final_counts['BROKEN']} unreadable={final_counts['UNREADABLE']} "
                       f"stuck={final_counts['STUCK']} repair_failures={len(failed_repairs)}")
print("done.")